Reference for this notebook: [geeksforgeeks](https://www.geeksforgeeks.org/nlp/next-word-prediction-with-deep-learning-in-nlp/)

Dataset :

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import regex as re # regular expression module

### Load and process data

In [ ]:
# Extract sentences and convert them to a list.
def file_to_sentence_list(file_path):
    with open(file_path, 'r') as file:
        text = file.read()
    sentences = []
    # Splitting the text into sentences using '.', '?', and '!'
    for sentence in re.split(r'(?<=[.!?])\s+', text):
        if sentence.strip():
            sentences.append(sentence.strip())
    return sentences

file_path = 'pizza.txt'
text_data = file_to_sentence_list(file_path)
print(text_data[:2])

['Pizza, the delectable and iconic dish that has transcended borders and captivated taste buds worldwide, is a testament to the extraordinary fusion of flavors, creativity, and cultural significance.', 'Originating from the sun-kissed lands of Italy, pizza has evolved into an art form that unites people from diverse backgrounds in a shared love for its mouthwatering combinations.']


In [ ]:
# Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(text_data)
print(tokenizer.word_index)
total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)


{'the': 1, 'and': 2, 'pizza': 3, 'of': 4, 'a': 5, 'to': 6, 'in': 7, 'has': 8, 'its': 9, 'for': 10, 'with': 11, 'it': 12, 'that': 13, 'is': 14, 'as': 15, 'culinary': 16, 'from': 17, 'become': 18, 'their': 19, 'have': 20, 'on': 21, 'flavors': 22, 'cheese': 23, 'toppings': 24, 'also': 25, 'delivery': 26, 'food': 27, 'people': 28, 'like': 29, 'world': 30, 'traditional': 31, 'made': 32, 'experience': 33, 'our': 34, 'pizzerias': 35, 'dish': 36, 'diverse': 37, 'crust': 38, 'delight': 39, 'symbol': 40, 'pizzas': 41, 'more': 42, 'making': 43, 'or': 44, 'iconic': 45, 'taste': 46, 'creativity': 47, 'cultural': 48, 'italy': 49, 'an': 50, 'combinations': 51, 'ancient': 52, 'who': 53, 'ingredients': 54, 'we': 55, 'even': 56, 'this': 57, 'style': 58, 'home': 59, 'indulgence': 60, 'beyond': 61, 'global': 62, 'inspired': 63, 'options': 64, 'those': 65, 'not': 66, 'but': 67, 'together': 68, 'allowing': 69, 'just': 70, 'comfort': 71, 'local': 72, 'may': 73, 'favorite': 74, 'will': 75, 'fusion': 76, 'into

In [ ]:
# Create input sequences
# [Pizza, the], [Pizza, the delectable] .....
input_sequences = []
for line in text_data:
  token_list = tokenizer.texts_to_sequences([line])[0]
  # token_list = tokenizer.texts_to_sequences([line])
  # print(token_list)
  # break
  for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    # print(n_gram_sequence)
    input_sequences.append(n_gram_sequence)
    # if (i == 4):
      # break
    # print(input_sequences)
  # break

In [ ]:
# Pad sequences and split into predictors and label
max_sequence_len = max([len(seq) for seq in input_sequences])
print(input_sequences)
input_sequences = np.array(pad_sequences(
	input_sequences, maxlen=max_sequence_len, padding='pre'))
X, y = input_sequences[:, :-1], input_sequences[:, -1]
print(X[0], y[0])
# Convert target data to one-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
print(X[0], y[0])

[[3, 1], [3, 1, 236], [3, 1, 236, 2], [3, 1, 236, 2, 45], [3, 1, 236, 2, 45, 36], [3, 1, 236, 2, 45, 36, 13], [3, 1, 236, 2, 45, 36, 13, 8], [3, 1, 236, 2, 45, 36, 13, 8, 115], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14, 5], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14, 5, 117], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14, 5, 117, 6], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14, 5, 117, 6, 1], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 239, 14, 5, 117, 6, 1, 240], [3, 1, 236, 2, 45, 36, 13, 8, 115, 237, 2, 238, 46, 116, 2

In [ ]:
# Define the model
model = Sequential()
model.add(Embedding(total_words, 10))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy',
			optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the model
model.fit(X, y, epochs=2, verbose=1)


Epoch 1/2
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.0441 - loss: 6.3657
Epoch 2/2
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.0492 - loss: 5.7864


In [ ]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 39, 10)         │         6,970 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 697)            │        89,913 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 504,155 (1.92 MB)

 Trainable params: 168,051 (656.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 336,104 (1.28 MB)

In [ ]:
# Generate next word predictions
seed_text = "Ram has a "

token_list = tokenizer.texts_to_sequences([seed_text])[0]
token_list = pad_sequences(
[token_list], maxlen=max_sequence_len-1, padding='pre')
predicted_probs = model.predict(token_list)
predicted_word = tokenizer.index_word[np.argmax(predicted_probs)]
# seed_text += " " + predicted_word

print("Next predicted words:", seed_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Next predicted words: Ram has a  pizza
